# Sales RAG with Context Engineering
Created 2025-06-30

This notebook demonstrates how to build a Retrieval-Augmented Generation (RAG) pipeline **tailored for sales enablement** that emphasizes **context engineering**—assembling rich multi-source context at run time—over simple prompt tweaks.

In [1]:
!pip -q install langchain langchain_community chromadb tiktoken openai python-dotenv

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
llama-index-core 0.12.30 requires SQLAlchemy[asyncio]>=1.4.49, but you have sqlalchemy 1.4.0 which is incompatible.
optuna 4.2.1 requires sqlalchemy>=1.4.2, but you have sqlalchemy 1.4.0 which is incompatible.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
spacy-pkuseg 1.0.0 requires numpy<3.0.0,>=2.0.0; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
magic-pdf 1.3.10 requires pdfminer.six==20250324, but you have pdfminer-six 20250327 which is incompatible.
datasets 2.21.0 requires fsspec[http]<=2024.6.1,>=2023.1.0, but you have fsspec 2025.3.2 which is incompatible.


In [ ]:

import os, getpass
from dotenv import load_dotenv
load_dotenv()
if not os.getenv('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('Enter your OPENAI_API_KEY: ')


## 1. Load & Chunk Sales Knowledge Base

In [ ]:

from pathlib import Path
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader, Docx2txtLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

data_dir = Path('sales_docs')
data_dir.mkdir(exist_ok=True)
print('Put your PDFs/DOCX/TXT into', data_dir.resolve())

loaders = []
for file in data_dir.iterdir():
    if file.suffix.lower()=='.pdf':
        loaders.append(PyPDFLoader(str(file)))
    elif file.suffix.lower() in ['.docx', '.doc']:
        loaders.append(Docx2txtLoader(str(file)))
    elif file.suffix.lower() in ['.txt', '.md']:
        loaders.append(TextLoader(str(file)))

raw_docs = []
for ld in loaders:
    raw_docs.extend(ld.load())
print(f'Loaded {len(raw_docs)} raw docs')

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=80)
documents = splitter.split_documents(raw_docs)
print('Chunks:', len(documents))


## 2. Create Embeddings & Vector Store

In [ ]:

from langchain.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
vector_store = Chroma.from_documents(documents, embeddings, collection_name='sales_knowledge')
retriever = vector_store.as_retriever()


## 3. Context Builder (Customer 360)

In [ ]:

from langchain.schema import Document
from typing import List, Dict

CUSTOMER_360: Dict[str,str] = {
    'Acme_Corp': 'Customer since 2019 | Industry: Logistics | ARR: $1.2M | Pain: last-mile delivery costs.'
}

def build_context(query:str, customer_id:str, stage:str='Discovery', k:int=4)->List[Document]:
    docs = retriever.get_relevant_documents(query, k=k)
    profile = Document(page_content=f'Customer profile ({customer_id})
Stage: {stage}
{CUSTOMER_360.get(customer_id,"N/A")}', metadata={'source':'crm'})
    return [profile]+docs


## 4. RAG Chain

In [ ]:

from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI

prompt = PromptTemplate(
    input_variables=['context','question'],
    template=("You are an expert sales engineer. Use only the context. If unsure, say you don't know.

Context:
{context}

Q: {question}
A:")
)
llm = ChatOpenAI(model_name='gpt-4o-mini', temperature=0.2)
rag_chain = RetrievalQA.from_chain_type(llm=llm, retriever=retriever, chain_type_kwargs={'prompt':prompt}, return_source_documents=True)


## 5. Demo Query

In [ ]:

query = 'Provide a battlecard snippet addressing route-optimization ROI for logistics.'
context_docs = build_context(query,'Acme_Corp','Solution Fit')
result = rag_chain({'query':query, 'context':'

'.join(d.page_content for d in context_docs)})
print(result['result'])
